# The Omnichannel Equilibrium: Causal MARL Architecture

First, let's install the necessary dependencies for Double Machine Learning, Reinforcement Learning, and Embeddings.

In [1]:
!pip install torch econml scikit-learn "ray[rllib]" gym

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 14.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 31.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached scipy-1.17.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 17.2 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requir

## 1. Feature Engineering & Embeddings

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

class EntityEmbeddingModel(nn.Module):
    """ Maps high-cardinality features like Brand and Color into dense latent spaces. """
    def __init__(self, cardinalities, embedding_dims):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=c, embedding_dim=d)
            for c, d in zip(cardinalities, embedding_dims)
        ])
        
    def forward(self, x):
        # x is a tensor of categorical indices
        embeds = [emb(x[:, i]) for i, emb in enumerate(self.embeddings)]
        return torch.cat(embeds, dim=1)

def compute_market_indices(df, short_window=7, long_window=30):
    """ Calculates Hype Index and prepares Channel Synergy Score foundations. """
    # Hype Index Calculation
    short_ema = df['Units_Sold'].ewm(span=short_window, adjust=False).mean()
    long_ema = df['Units_Sold'].ewm(span=long_window, adjust=False).mean()
    df['Hype_Index'] = short_ema / (long_ema + 1e-5)
    return df

## 2. Causal Inference (Double ML)

In [3]:
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor

class CausalPriceElasticityModel:
    """ Uses Double Machine Learning to isolate the CATE of Price on Units Sold. """
    def __init__(self):
        # Using Random Forests to model nuisance functions (E[Y|X], E[T|X])
        self.dml = LinearDML(
            model_y=RandomForestRegressor(n_estimators=100, max_depth=5),
            model_t=RandomForestRegressor(n_estimators=100, max_depth=5),
            discrete_treatment=False,
            cv=3
        )
        
    def fit(self, Y, T, X):
        """
        Y: Units_Sold | T: Price_USD | X: Confounders (Embeddings, Seasonality, etc.)
        """
        self.dml.fit(Y, T, X=X)
        
    def estimate_elasticity(self, X):
        # Returns the Conditional Average Treatment Effect (CATE)
        return self.dml.effect(X)

/Users/tiz/DataScience/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-24 10:46:02,826	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


## 3. Market Simulator / World Model

In [4]:
class MarketWorldModel:
    """ Simulates market environment dynamics for the RL agents. """
    def __init__(self, causal_model, baseline_demand_model):
        self.causal_model = causal_model
        self.baseline_demand = baseline_demand_model
        
    def simulate_demand(self, state_features, action_price_adjustment):
        """ Projects demand given the current state and the agent's price action. """
        # Base demand under status-quo conditions
        base_demand = self.baseline_demand.predict(state_features)
        
        # Causal impact of the price adjustment
        elasticity = self.causal_model.estimate_elasticity(state_features)
        
        # New demand = Base Demand + (Elasticity * Price Change) + Stochastic Reality Noise
        noise = np.random.normal(0, 0.1 * base_demand)
        simulated_demand = base_demand + (elasticity * action_price_adjustment) + noise
        
        return np.maximum(0, simulated_demand)

## 4. MARL Environment (Ray RLlib)

In [5]:
import ray
from ray.rllib.env.multi_agent_env import MultiAgentEnv
from gym.spaces import Box

class OmnichannelEnv(MultiAgentEnv):
    """ The Multi-Agent RL Environment managing global inventory and pricing. """
    def __init__(self, config):
        super().__init__()
        self.agents = config.get("agents", ["UK_Online", "UK_Retail", "USA_Online"])
        self.world_model = config["world_model"]
        
        # Continuous action: [-1.0, 1.0] representing +/- % price change from baseline
        self.action_space = Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
        
        # State observation dimension depends on concatenated embedding/feature length
        self.observation_space = Box(low=-np.inf, high=np.inf, shape=(20,), dtype=np.float32)
        
        self._agent_ids = set(self.agents)
        self.inventory = {agent: 1000 for agent in self.agents} # Mock initial inventory
        
    def reset(self, *, seed=None, options=None):
        self.inventory = {agent: 1000 for agent in self.agents}
        obs = {agent: np.zeros(20, dtype=np.float32) for agent in self.agents}
        return obs, {}

    def step(self, action_dict):
        obs, rewards, terminateds, truncateds, infos = {}, {}, {}, {}, {}
        
        for agent_id, action in action_dict.items():
            price_adj = action[0]
            current_state = np.zeros(20) # Extracted from state pipeline
            
            # 1. World Model Simulates Demand
            demand = self.world_model.simulate_demand([current_state], price_adj)[0]
            
            # 2. Inventory Dynamics
            sold = min(demand, self.inventory[agent_id])
            self.inventory[agent_id] -= sold
            
            # 3. Calculate Revenue & Penalties
            base_price = 100.0 # Extracted from baseline data
            actual_price = base_price * (1 + price_adj)
            revenue = sold * actual_price
            
            stockout_penalty = 10.0 * max(0, demand - self.inventory[agent_id])
            holding_cost = 0.5 * self.inventory[agent_id]
            
            # 4. Reward Formulation
            rewards[agent_id] = revenue - stockout_penalty - holding_cost
            
            # Next state update
            obs[agent_id] = np.random.randn(20).astype(np.float32)
            terminateds[agent_id] = False
            truncateds[agent_id] = False
            
        terminateds["__all__"] = False
        truncateds["__all__"] = False
        
        return obs, rewards, terminateds, truncateds, infos

2026-04-24 10:46:05,868	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## 5. Execution Entrypoint

In [6]:
class MockBaselineDemandModel:
    def predict(self, state_features):
        return np.array([50.0])

class MockCausalModel:
    def estimate_elasticity(self, state_features):
        return np.array([-10.0]) # Example elasticity

# Start Ray
ray.init(ignore_reinit_error=True)

# Pre-train CausalPriceElasticityModel and Demand model in reality
mock_world_model = MarketWorldModel(causal_model=MockCausalModel(), baseline_demand_model=MockBaselineDemandModel())

config = {
    "env": OmnichannelEnv,
    "env_config": {
        "agents": ["UK_Online", "USA_Mall", "FR_Retail"],
        "world_model": mock_world_model
    },
    "framework": "torch",
}

print("Initializing The Omnichannel Equilibrium MARL using Ray RLlib...")
# To train, you would use:
# from ray.rllib.algorithms.sac import SAC
# trainer = SAC(config=config)
# trainer.train()
print("Environment created and configured successfully!")

2026-04-24 10:46:09,768	INFO worker.py:2012 -- Started a local Ray instance.


Initializing The Omnichannel Equilibrium MARL using Ray RLlib...
Environment created and configured successfully!


/Users/tiz/DataScience/.venv/lib/python3.14/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
